# 3교시 · 데이터 결합과 집계
### — 여러 표를 하나로 합쳐 요약하기

앞 시간에 필요한 것만 남겼습니다. 이제 그것을 **묶어서 요약**합니다.
엑셀의 **피벗테이블**과 **VLOOKUP** 에 해당하는 부분입니다.

**이 시간이 끝나면 할 수 있는 것**

1. 항목별로 묶어서 합계·평균을 낼 수 있다
2. 두 개의 표를 이어 붙일 수 있다
3. 비교 기준을 바꿔 가며 볼 수 있다
4. **같은 데이터로 정반대 결론이 나올 수 있다는 것을 안다**

In [ ]:
import pandas as pd

BASE = 'https://raw.githubusercontent.com/JasonWhiteLee/ak-data-analysis-basics/main/'

orders  = pd.read_csv(BASE + 'superstore_orders.csv', parse_dates=['Order Date', 'Ship Date'])
returns = pd.read_csv(BASE + 'superstore_returns.csv')
people  = pd.read_csv(BASE + 'superstore_people.csv')

# 2교시에서 배운 대로 중복부터 제거하고 시작합니다
orders = orders.drop_duplicates()

print(orders.shape)
orders.head(3)

---
# 3-1. 시작하기 전에 — "매출 3억"

회의에서 이런 보고를 들었다고 해 봅시다.

> **"이번 분기 매출은 3억입니다."**

이 말을 듣고 무슨 생각이 드나요?

**아무 생각도 들지 않는 것이 정상입니다.** 좋은 건지 나쁜 건지 알 수 없기 때문입니다.

- 지난 분기가 2억이었다면 -> 좋은 소식
- 지난 분기가 5억이었다면 -> 나쁜 소식
- 목표가 4억이었다면 -> 미달

숫자 하나만으로는 판단할 수 없습니다. **무언가와 견주어야 의미가 생깁니다.**

이번 시간에 하는 일은 두 가지입니다.
1. 견줄 수 있는 형태로 **묶어서 요약**하기 (집계)
2. 필요한 정보를 다른 표에서 **가져와 붙이기** (결합)

---
# 3-2. groupby — 엑셀 피벗테이블

`groupby` 는 "이 열의 값이 같은 것끼리 묶어라"는 뜻입니다.
묶은 다음에는 **무엇을 계산할지** 알려 줘야 합니다.

```
orders.groupby('무엇으로묶을까')['무엇을계산할까'].어떻게()
```

In [ ]:
orders.groupby('Category')['Sales'].sum()

방금 만든 것이 엑셀 피벗테이블과 똑같습니다.

| 엑셀 | pandas |
|---|---|
| 행 영역에 `Category` | `groupby('Category')` |
| 값 영역에 `Sales` | `['Sales']` |
| 값 요약 = 합계 | `.sum()` |

## 계산 방법 바꾸기

In [ ]:
print('합계'); print(orders.groupby('Category')['Sales'].sum().round(0))
print()
print('평균'); print(orders.groupby('Category')['Sales'].mean().round(0))
print()
print('건수'); print(orders.groupby('Category')['Sales'].count())

## 한 번에 여러 개 — `.agg()`

매번 따로 부르지 않고 한 번에 볼 수 있습니다.

In [ ]:
orders.groupby('Category')['Sales'].agg(['sum', 'mean', 'count']).round(0)

## 열마다 다른 계산

`Sales` 는 합계, `Profit` 은 합계, `Order ID` 는 개수 — 이렇게 섞을 수도 있습니다.

In [ ]:
요약 = orders.groupby('Category').agg(
    매출=('Sales', 'sum'),
    이익=('Profit', 'sum'),
    건수=('Order ID', 'nunique'),
).round(0)

요약

## 여기서 하나 계산해 봅시다 — 이익률

In [ ]:
요약['이익률(%)'] = (요약['이익'] / 요약['매출'] * 100).round(1)

요약

## 결과를 보세요

**매출 1위는 Furniture 입니다.** (858,518)
매출만 보면 "가구가 제일 잘 팔린다"가 됩니다.

그런데 이익률을 보세요.

| Category | 매출 | 이익 | 이익률 |
|---|---|---|---|
| **Furniture** | **858,518** | 19,730 | **2.3%** |
| Office Supplies | 740,730 | 126,023 | 17.0% |
| Technology | 839,192 | 146,543 | 17.5% |

**매출 1위인 Furniture 의 이익률이 가장 낮습니다.**
이익 금액으로 보면 Technology 의 7분의 1 수준입니다.

> 매출만 요약해서 보고했다면 "가구가 효자 품목"이 되고,
> 이익을 요약해서 보고했다면 "가구는 남는 게 없다"가 됩니다.
> **둘 다 같은 데이터에서 나온 사실입니다.**
> 어떤 값을 요약하느냐가 결론을 정합니다.

---
# 3-3. 두 가지 기준으로 묶기

`groupby` 에 열을 두 개 넣으면 2단으로 묶입니다.

In [ ]:
orders.groupby(['Region', 'Category'])['Sales'].sum().round(0)

결과가 세로로 길게 나옵니다. 읽기 불편하죠.
**행과 열로 펼치면** 훨씬 잘 보입니다.

## pivot_table — 표로 펼치기

In [ ]:
orders.pivot_table(
    index='Region',
    columns='Category',
    values='Sales',
    aggfunc='sum',
).round(0)

| 인자 | 뜻 | 엑셀 피벗 |
|---|---|---|
| `index` | 행에 무엇을 | 행 영역 |
| `columns` | 열에 무엇을 | 열 영역 |
| `values` | 무슨 값을 | 값 영역 |
| `aggfunc` | 어떻게 요약 | 값 요약 기준 |

## 이번엔 이익으로 바꿔 봅시다

`values` 만 `Profit` 으로 바꿉니다.

In [ ]:
orders.pivot_table(
    index='Region',
    columns='Category',
    values='Profit',
    aggfunc='sum',
).round(0)

## 음수가 하나 있습니다

**Central 지역의 Furniture 가 -2,802 입니다.**
그 지역 그 품목은 팔수록 손해를 보고 있습니다.

매출 표에서는 전혀 보이지 않던 사실입니다.
**쪼개서 봐야 나오는 것**이 있습니다.

> 전체 이익만 봤다면 "이익 잘 나고 있다"로 끝났을 겁니다.
> 지역별로만 봤어도, 품목별로만 봤어도 안 보였습니다.
> **두 기준을 교차해서 봐야 나왔습니다.**

---
# 3-4. 기준선 — 같은 데이터, 반대 결론

이제 3-1 에서 던진 질문으로 돌아갑니다.
**무언가와 견주어야 의미가 생긴다**고 했습니다. 그럼 무엇과 견줄까요?

월별 매출을 만들어 보겠습니다.

In [ ]:
orders['연월'] = orders['Order Date'].dt.to_period('M')

월매출 = orders.groupby('연월')['Sales'].sum()

월매출.tail(13).round(0)

## 견주는 방법 두 가지

같은 매출에 **다른 기준**을 대 봅니다.

- **전월 대비** — 바로 앞 달과 비교. `pct_change()`
- **전년 동월 대비** — 1년 전 같은 달과 비교. `pct_change(12)`

In [ ]:
추이 = pd.DataFrame({'매출': 월매출})

추이['전월대비(%)']    = 추이['매출'].pct_change() * 100
추이['전년동월대비(%)'] = 추이['매출'].pct_change(12) * 100

추이.loc['2026-07':'2026-12'].round(1)

## 2026년 10월을 보세요

| | 값 |
|---|---|
| 매출 | 83,363 |
| **전월 대비** | **-5.3%** |
| **전년 동월 대비** | **+39.3%** |

**같은 달, 같은 매출입니다.** 그런데 보고서 문장은 이렇게 달라집니다.

> "10월 매출은 전월 대비 **5.3% 감소**했습니다."
>
> "10월 매출은 전년 동월 대비 **39.3% 증가**했습니다."

둘 다 사실입니다. **어느 쪽도 거짓말이 아닙니다.**

> ### 여기서 기억할 것
> 비교 기준을 고르는 순간, 결론의 방향이 상당 부분 정해집니다.
> 그래서 **어떤 기준으로 비교했는지를 반드시 함께 적어야 합니다.**
>
> "매출이 늘었다" 가 아니라 **"전년 동월 대비 39.3% 늘었다"** 라고 적는 이유입니다.

## 1월은 더 심합니다

In [ ]:
추이.loc['2026-01':'2026-03'].round(1)

2026년 1월 — 전월 대비 **-54.8%**, 전년 동월 대비 **+75.7%** 입니다.

전월 대비가 크게 떨어진 이유는 짐작할 수 있습니다.
**12월은 연말이라 원래 많이 팔립니다.** 1월과 12월을 비교하는 것 자체가 공정하지 않습니다.

> 계절성이 있는 데이터에서는 전월 대비가 오해를 부릅니다.
> 이럴 때 전년 동월 대비를 쓰는 것입니다.
> **기준선을 고르는 데에도 근거가 필요합니다.**

## 세 번째 기준 — 전체 평균과 비교

기준은 "지난 달"이나 "작년"만 있는 게 아닙니다.

In [ ]:
평균 = 월매출.mean()

비교 = pd.DataFrame({'매출': 월매출.loc['2026-01':'2026-12']})
비교['평균대비(%)'] = ((비교['매출'] - 평균) / 평균 * 100).round(1)

print('전체 기간 월평균: {:,.0f}'.format(평균))
비교.round(0)

> 같은 12개월을 세 가지 기준으로 봤습니다. 셋 다 다른 이야기를 합니다.
> **"어느 기준이 맞는가"에 정답은 없습니다.** 무엇을 판단하려는지에 따라 고릅니다.
> 다만 **고른 기준을 밝혀야** 상대방이 그 판단을 검증할 수 있습니다.

---
# 3-5. merge — 두 표를 이어 붙이기

**엑셀의 VLOOKUP** 에 해당합니다.

지금 `orders` 에는 반품 정보가 없습니다. 그건 `returns` 라는 별도의 표에 있습니다.

In [ ]:
print('orders :', orders.shape)
print('returns:', returns.shape)

returns.head(3)

두 표에 **공통으로 있는 열**이 `Order ID` 입니다. 이걸 열쇠 삼아 붙입니다.

In [ ]:
합친표 = orders.merge(returns, on='Order ID', how='left')

print('붙이기 전:', len(orders))
print('붙인 후  :', len(합친표))

합친표[['Order ID', 'Sales', 'Returned']].head(3)

## `how` 가 무엇을 남길지 정합니다

| `how` | 무엇이 남나 | 엑셀로 치면 |
|---|---|---|
| `'left'` | **왼쪽 표는 전부**, 오른쪽은 맞는 것만 | VLOOKUP |
| `'inner'` | **양쪽 다 있는 것만** | 교집합 |
| `'outer'` | 양쪽 전부 | 합집합 |
| `'right'` | 오른쪽 표 전부 | 거꾸로 VLOOKUP |

`how='left'` 를 썼기 때문에 주문은 하나도 안 사라졌습니다.
반품 기록이 없는 주문은 `Returned` 가 **빈칸**이 됩니다.

In [ ]:
print('Returned 빈칸:', 합친표['Returned'].isna().sum())

# 빈칸 = 반품 안 됨 이므로 'No' 로 채웁니다
합친표['Returned'] = 합친표['Returned'].fillna('No')

합친표['Returned'].value_counts()

> ### 여기서 `fillna` 를 쓴 이유
> 2교시에서 "빈칸을 함부로 채우지 말라"고 했습니다.
> 그런데 여기서는 채웠습니다. **왜 비었는지 알기 때문입니다.**
>
> 반품 표에 없다는 것은 반품되지 않았다는 뜻입니다.
> **의미를 아는 빈칸은 채워도 됩니다.** 모르는 빈칸이 위험한 것입니다.

## merge 후에는 반드시 행 수를 확인합니다

붙였는데 행이 **늘어났다면** 오른쪽 표에 같은 열쇠가 여러 개 있다는 뜻입니다.
붙였는데 값이 **다 비었다면** 열쇠가 안 맞는다는 뜻입니다 (공백, 대소문자, 타입 차이).

In [ ]:
print('행 수 변화 :', len(orders), '->', len(합친표))
print('매출 합 변화: {:,.0f} -> {:,.0f}'.format(orders['Sales'].sum(), 합친표['Sales'].sum()))

## 작은 표도 붙여 봅시다 — 지역 담당자

In [ ]:
people

In [ ]:
합친표 = 합친표.merge(people, on='Region', how='left')

합친표[['Region', 'Regional Manager', 'Sales']].head(3)

## 이제 담당자별 실적을 볼 수 있습니다

원래 `orders` 에는 없던 정보입니다. 붙였기 때문에 생긴 질문입니다.

In [ ]:
합친표.groupby('Regional Manager').agg(
    매출=('Sales', 'sum'),
    이익=('Profit', 'sum'),
).round(0).sort_values('매출', ascending=False)

---
# 3-6. 반품률 — 집계와 결합을 함께 써 보기

앞에서 만든 `Returned` 열로 지역별 반품률을 내 봅니다.

In [ ]:
주문단위 = 합친표.groupby('Order ID').agg(
    Region=('Region', 'first'),
    Category=('Category', 'first'),
    Sales=('Sales', 'sum'),
    반품=('Returned', 'first'),
)

주문단위['반품'] = (주문단위['반품'] == 'Yes')

print('주문 건수:', len(주문단위))
주문단위.head(3)

> ### 왜 `groupby('Order ID')` 를 먼저 했을까요
> 1교시에서 확인했듯이 **한 줄은 주문이 아니라 품목**입니다.
> 품목 단위로 반품률을 세면, 물건을 많이 담은 주문이 여러 번 세어집니다.
>
> **무엇을 세는지에 맞춰 단위를 먼저 맞춰야 합니다.**

In [ ]:
(주문단위.groupby('Region')['반품'].mean() * 100).round(2)

## West 지역이 눈에 띕니다

다른 지역은 3% 안팎인데 **West 만 11.6%** 입니다. 세 배가 넘습니다.

여기서 바로 "West 에 문제가 있다"고 결론 내리고 싶어집니다.
하지만 지금 알 수 있는 것은 **숫자가 다르다는 사실뿐**입니다.

- 정말 West 의 반품이 많은 걸까요?
- 아니면 West 만 반품 기록을 꼼꼼히 남기는 걸까요?
- 아니면 West 가 반품 많은 품목을 주로 파는 걸까요?

> **이 차이가 의미 있는 차이인지 판단하는 방법은 6교시에 다룹니다.**
> 지금은 "차이를 발견했다"까지입니다.

---
# 3-7. 실습 — 같은 데이터로 두 개의 보고서 쓰기

아래 셀을 실행해서 **2026년 10월 실적 보고**에 쓸 재료를 만드세요.

In [ ]:
목표월 = '2026-10'

print('=== 2026년 10월 ===')
print('매출          : {:>12,.0f}'.format(추이.loc[목표월, '매출']))
print('전월 대비     : {:>+11.1f}%'.format(추이.loc[목표월, '전월대비(%)']))
print('전년 동월 대비: {:>+11.1f}%'.format(추이.loc[목표월, '전년동월대비(%)']))
print('월평균 대비   : {:>+11.1f}%'.format((추이.loc[목표월, '매출'] - 평균) / 평균 * 100))

## 여기에 적으세요

**이 텍스트 셀을 더블클릭**해서 채우세요.

### 보고서 A — 이 달이 좋았다고 말하는 한 문장
(어떤 기준을 골랐는지 반드시 포함해서)


### 보고서 B — 이 달이 나빴다고 말하는 한 문장
(마찬가지로 기준 포함)


### 둘 중 어느 쪽에도 거짓말이 있습니까


### 그럼 상사에게는 어떻게 보고하는 것이 맞을까요


---

> ### 생각해 볼 것
> 위 두 문장 모두 사실입니다. 그런데 읽는 사람은 완전히 다르게 받아들입니다.
>
> **하나의 기준만 골라서 보고하면, 고른 사람의 의도가 결론에 섞여 들어갑니다.**
> 그래서 실무에서는 보통 **기준을 두 개 이상 함께** 적습니다.

## 추가 실습 — 적자 품목 찾기

`Region` 과 `Sub-Category` 로 교차해서 **이익이 음수인 칸**을 찾아보세요.

In [ ]:
적자표 = orders.pivot_table(
    index='Sub-Category',
    columns='Region',
    values='Profit',
    aggfunc='sum',
).round(0)

# 음수가 하나라도 있는 품목만
적자표[(적자표 < 0).any(axis=1)]

### 여기에 적으세요

**가장 손실이 큰 조합 하나**를 고르고, 아래 질문에 답하세요.

- 어느 지역, 어느 품목입니까
- 이 사실은 전체 이익만 봤을 때 보였습니까
- 이걸 알았다면 **무엇을 다르게 하겠습니까**

> 마지막 질문이 중요합니다.
> 아무 행동도 바뀌지 않는다면, 이 분석은 하지 않아도 됐던 것입니다.

---
# 정리 — 오늘 쓴 것

## 코드

| 하는 일 | 코드 |
|---|---|
| 묶어서 합계 | `df.groupby('열')['값'].sum()` |
| 여러 계산 한 번에 | `df.groupby('열')['값'].agg(['sum','mean','count'])` |
| 열마다 다른 계산 | `df.groupby('열').agg(이름=('값','sum'))` |
| 두 기준으로 묶기 | `df.groupby(['열1','열2'])` |
| 표로 펼치기 | `df.pivot_table(index=, columns=, values=, aggfunc=)` |
| 표 잇기 | `df.merge(다른표, on='열쇠', how='left')` |
| 월 만들기 | `df['날짜'].dt.to_period('M')` |
| 전월 대비 | `.pct_change()` |
| 전년 동월 대비 | `.pct_change(12)` |

## 남길 것 세 가지

1. **숫자 하나만으로는 판단할 수 없다** — 무언가와 견주어야 의미가 생깁니다
2. **기준을 고르면 결론의 방향이 정해진다** — 10월은 -5.3% 이기도 하고 +39.3% 이기도 했습니다
3. **쪼개야 보이는 것이 있다** — Central 지역 Furniture 는 적자였습니다

---

### 다음 시간

지금까지 계속 **합계와 평균**을 봤습니다.
다음 시간에는 이렇게 묻습니다 — **그 평균, 믿어도 됩니까?**